# Brazil Occupation Pay Anatomy

Folha de S.Paulo published a story on 20 July 2026 about the highest-paid occupations in Brazil. This notebook uses that story as a starting point, then slows the ranking down.

The question is not only who ranks first. The deeper question is what kind of wage architecture sits behind each occupation: a high middle, a long top tail, narrow access, or large dispersion inside the same occupational label.

This is a descriptive project using PNAD Continua/IBGE Q1 2026 microdata. It does not claim causality or prove discrimination. It shows where the ranking becomes more interesting when we inspect distributions, representation and inequality decomposition.


## How To Reproduce

Run this from the project root. Raw PNAD files are cached outside the repo by default.

```powershell
Rscript requirements.R
Rscript R/run_analysis.R
Rscript R/run_deep_anatomy.R
Rscript R/render_deep_figures.R
Rscript R/validate_outputs.R
```


In [1]:
library(readr)
library(jsonlite)
library(dplyr)

summary <- fromJSON("../data/processed/analysis_summary.json")
cat(sprintf("Run generated at: %s\n", summary$generated_at))
cat(sprintf("Rows imported: %s\n", format(summary$rows_imported, big.mark = ",")))
cat(sprintf("Analytic rows: %s\n", format(summary$rows_analytic, big.mark = ",")))
cat(sprintf("Occupations found: %s\n", summary$occupations_total))
cat(sprintf("Occupations after reliability filter: %s\n", summary$occupations_after_filter))
cat(sprintf("Minimum occupation sample: %s\n", summary$min_sample))
cat(sprintf("Minimum occupation-sex cell sample: %s\n", summary$min_cell_sample))


Run generated at: 2026-07-20 16:02:45 -03
Rows imported: 511,149
Analytic rows: 223,762
Occupations found: 430
Occupations after reliability filter: 266
Minimum occupation sample: 80
Minimum occupation-sex cell sample: 40


## Ranking Result

The first result already changes the story a bit. `Diretores gerais e gerentes gerais` lead by mean income, but their median is far lower than the mean. That points to a stretched distribution inside the occupation.

Among `Medicos especialistas`, the mean is lower than the directors group, but the median is higher. That is a different kind of high-income occupation: less about a huge top tail, more about a high middle.

| Rank | Occupation | Mean income | Weighted median | Sample n |
| --- | --- | --- | --- | --- |
| 1 | 1120 - Diretores gerais e gerentes gerais | R$ 22.416 | R$ 11.000 | 219 |
| 2 | 2212 - Médicos especialistas | R$ 19.293 | R$ 15.000 | 837 |
| 3 | 3355 - Inspetores de polícia e detetives | R$ 14.252 | R$ 11.000 | 176 |
| 4 | 0411 - Oficiais de polícia militar | R$ 13.924 | R$ 12.300 | 91 |
| 5 | 2619 - Profissionais em direito não classificados anteriormente | R$ 13.704 | R$ 12.000 | 447 |
| 6 | 1211 - Dirigentes financeiros | R$ 13.693 | R$ 8.000 | 246 |
| 7 | 1330 - Dirigentes de serviços de tecnologia da informação e comunicações | R$ 13.570 | R$ 10.000 | 250 |
| 8 | 1349 - Dirigentes e gerentes de serviços profissionais não classificados anteriormente | R$ 12.840 | R$ 8.000 | 127 |
| 9 | 2211 - Médicos gerais | R$ 12.551 | R$ 12.000 | 544 |
| 10 | 1323 - Dirigentes de empresas de construção | R$ 12.483 | R$ 10.000 | 133 |


![Top occupations by income](../figures/top20_occupations_income.png)


In [2]:
occupation_rank <- read_csv("../data/processed/occupation_rank_q1_2026.csv", show_col_types = FALSE)
occupation_rank |>
  select(rank, occupation_label, avg_income, median_income, sample_n) |>
  slice_head(n = 10)


| Rank | Occupation | Mean income | Weighted median | Sample n |
| --- | --- | --- | --- | --- |
| 1 | 1120 - Diretores gerais e gerentes gerais | R$ 22.416 | R$ 11.000 | 219 |
| 2 | 2212 - Médicos especialistas | R$ 19.293 | R$ 15.000 | 837 |
| 3 | 3355 - Inspetores de polícia e detetives | R$ 14.252 | R$ 11.000 | 176 |
| 4 | 0411 - Oficiais de polícia militar | R$ 13.924 | R$ 12.300 | 91 |
| 5 | 2619 - Profissionais em direito não classificados anteriormente | R$ 13.704 | R$ 12.000 | 447 |
| 6 | 1211 - Dirigentes financeiros | R$ 13.693 | R$ 8.000 | 246 |
| 7 | 1330 - Dirigentes de serviços de tecnologia da informação e comunicações | R$ 13.570 | R$ 10.000 | 250 |
| 8 | 1349 - Dirigentes e gerentes de serviços profissionais não classificados anteriormente | R$ 12.840 | R$ 8.000 | 127 |
| 9 | 2211 - Médicos gerais | R$ 12.551 | R$ 12.000 | 544 |
| 10 | 1323 - Dirigentes de empresas de construção | R$ 12.483 | R$ 10.000 | 133 |

## Deep Wage Anatomy

The ranking becomes more useful when we separate average income from the shape of the distribution. The map below compares each occupation's weighted median with its mean/median ratio.

A high X value means the typical worker is better paid. A high Y value means the average is being pulled away from the middle by the upper tail.

![Wage anatomy map](../figures/wage_anatomy_map.png)


## Distribution Ladders

This figure shows the P10-P90 range for selected occupations. The green dot is the median and the gold dot is the mean.

The visual point is simple: two occupations can both be near the top of a ranking while having very different internal structures.

![Distribution ladder](../figures/distribution_ladder_selected_occupations.png)


## Inequality Decomposition

The Theil decomposition is the strongest upgrade in the project. It asks how much observed earnings inequality is between groups and how much remains inside the groups.

In this run, occupation explains about **42.6%** of Theil inequality, while **57.4%** remains inside occupations. Education explains about **27.3%** between groups. State explains about **5.3%** between groups.

That means occupational sorting matters a lot, but the occupation label is still too broad to explain the full wage structure.

![Theil inequality decomposition](../figures/theil_inequality_decomposition.png)


## Composition Inside The Ranking

The top of the income ranking is not only about occupation names. It also reflects who is concentrated inside each occupation. The chart below shows the estimated gender composition among the 15 occupations with highest mean income.

![Gender composition](../figures/top15_gender_composition.png)


## Gender Gap Sensitivity

The table below only includes occupation-sex cells with enough unweighted observations. These are descriptive differences in survey-weighted mean income; they should not be read as causal evidence.

| Occupation | Men mean | Women mean | Women vs men |
| --- | --- | --- | --- |
| 1120 - Diretores gerais e gerentes gerais | R$ 26.712 | R$ 12.656 | -52,6% |
| 1211 - Dirigentes financeiros | R$ 17.345 | R$ 9.970 | -42,5% |
| 1221 - Dirigentes de vendas e comercialização | R$ 13.490 | R$ 9.456 | -29,9% |
| 1346 - Gerentes de sucursais de bancos, de serviços financeiros e de seguros | R$ 13.268 | R$ 9.528 | -28,2% |
| 1330 - Dirigentes de serviços de tecnologia da informação e comunicações | R$ 14.383 | R$ 10.459 | -27,3% |


In [3]:
gender_gap <- read_csv("../data/processed/top15_gender_gap.csv", show_col_types = FALSE)
gender_gap |>
  select(occupation_label, avg_income_Homem, avg_income_Mulher, gap_women_vs_men) |>
  slice_head(n = 5)


| Occupation | Men mean | Women mean | Women vs men |
| --- | --- | --- | --- |
| 1120 - Diretores gerais e gerentes gerais | R$ 26.712 | R$ 12.656 | -52,6% |
| 1211 - Dirigentes financeiros | R$ 17.345 | R$ 9.970 | -42,5% |
| 1221 - Dirigentes de vendas e comercialização | R$ 13.490 | R$ 9.456 | -29,9% |
| 1346 - Gerentes de sucursais de bancos, de serviços financeiros e de seguros | R$ 13.268 | R$ 9.528 | -28,2% |
| 1330 - Dirigentes de serviços de tecnologia da informação e comunicações | R$ 14.383 | R$ 10.459 | -27,3% |

## Less Obvious Findings

- Occupation accounts for about **42.6%** of measured Theil inequality, but **57.4%** remains inside occupations.
- `Diretores gerais e gerentes gerais` are a long-tail elite occupation: high mean, much lower median and high within-occupation dispersion.
- `Medicos especialistas` are closer to a high-middle occupation: still unequal, but less dependent on the upper tail.
- Some low-income occupations also have long tails, which means tail analysis is not only about elite jobs.
- High-income, narrow-access occupations include top management and technology leadership roles with low female and preta/parda/indigena representation quotients.
- Presence and parity differ. Some occupations have substantial female participation while still showing large descriptive within-occupation gender gaps.


## Access Anatomy

The next layer asks who is over- or under-represented in each occupation. Location quotients compare each occupation with the overall analytic sample.

A value above 1 means over-representation. A value below 1 means under-representation.

![Access representation quadrants](../figures/access_representation_quadrants.png)


## Presence Is Not Parity

Female participation and within-occupation pay gaps are related, but they are not the same question. Some occupations have meaningful female presence and still show large descriptive gaps in mean income.

![Access versus internal gender gap](../figures/access_vs_internal_gender_gap.png)


## Interpretation

This is the point I would keep from the notebook: a wage ranking is not a ladder; it is a set of hidden wage architectures.

Some occupations are high because the typical worker is well paid. Some are high because the upper tail pulls the average upward. Some have better representation than expected; others look like narrow doors into high-income work.

The gender and race/color estimates are descriptive gaps, not causal effects. PNAD does not observe same firm, exact job title, seniority, bonus structure or productivity. The responsible conclusion is more precise: these patterns show where deeper questions are worth asking.
